In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))




# import libraries 
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# Load and create the data
train_csv = '/kaggle/input/competitions/playground-series-s6e5/train.csv'
train_data = pd.read_csv(train_csv)
test_csv = '/kaggle/input/competitions/playground-series-s6e5/test.csv'
test_data = pd.read_csv(test_csv)
#print(train_data.head())

# Make column which calculates the gaps between next and rear driver
#def calculate_gaps(df):
#    # 1. Sort data by Race, Year, Round and Position
#    # This puts the drivers in the order they move around the track.
#    df = df.sort_values(by=['Race', 'Year', 'LapNumber', 'Position']).reset_index(drop=True)
#    
#    # 2. We calculate the time of the NEXT driver (Position - 1)
#    # shift(1) takes the value of the previous row
#    df['Front_Driver_LapTime'] = df.groupby(['Race', 'Year', 'LapNumber'])['LapTime (s)'].shift(1)
#    
#    # 3. We calculate the time of the REAR driver (Position + 1)
#    # shift(1) takes the value of the next row
#    df['Back_Driver_LapTime'] = df.groupby(['Race', 'Year', 'LapNumber'])['LapTime (s)'].shift(-1)
#    
#   # 4. Find the gap in sec
#    df['Gap_To_Front'] = df['LapTime (s)'] - df['Front_Driver_LapTime']
#    df['Gap_To_Back'] = df['Back_Driver_LapTime'] - df['LapTime (s)']
#    
#    # 5. The first (Position 1) has no leading and the last has no trailing.
#    # Replace the NaNs generated with a large number (0 seconds)
#    df['Gap_To_Front'] = df['Gap_To_Front'].fillna(0)
#    df['Gap_To_Back'] = df['Gap_To_Back'].fillna(0)
#    
#    # Delete the intermediate columns
#    df = df.drop(['Front_Driver_LapTime', 'Back_Driver_LapTime'], axis=1)
#    
#    return df


# Make "get_expected_life", a function for the expected life of tires
#def get_expected_life(compound):
#    if compound == 'HARD': return 40
#    if compound == 'MEDIUM': return 25
#    if compound == 'SOFT': return 15
#    return 25 # A mean value if there is another compound (π.χ. Intermediate/Wet)


# Keep only lines with Lap Time < 200s
#train_data = train_data[train_data['LapTime (s)'] < 200]
# Add 'calculate_gaps' column
#train_data = calculate_gaps(train_data)
#test_data = calculate_gaps(test_data)

# Add 'Tyre_Life_Pct' column
#train_data['Tyre_Life_Pct'] = train_data['TyreLife'] / train_data['Compound'].apply(get_expected_life)
#test_data['Tyre_Life_Pct'] = test_data['TyreLife'] / test_data['Compound'].apply(get_expected_life)
train_data['Tyre_vs_Lap'] = (train_data['TyreLife'] * train_data['RaceProgress'] * train_data['LapNumber']) ** 0.33
test_data['Tyre_vs_Lap'] = (test_data['TyreLife'] * test_data['RaceProgress'] * test_data['LapNumber']) ** 0.33
train_data['Degr_vs_Time'] = (train_data['LapTime (s)'] * train_data['LapTime_Delta'] * train_data['Cumulative_Degradation']) ** 0.33
test_data['Degr_vs_Time'] = (test_data['LapTime (s)'] * test_data['LapTime_Delta'] * test_data['Cumulative_Degradation']) ** 0.33
train_data['Delta_to_Degr'] = train_data['LapTime_Delta'] / train_data['Cumulative_Degradation']
test_data['Delta_to_Degr'] = test_data['LapTime_Delta'] / test_data['Cumulative_Degradation']
train_data['Time_to_Degr'] = train_data['LapTime (s)'] / train_data['Cumulative_Degradation']
test_data['Time_to_Degr'] = test_data['LapTime (s)'] / test_data['Cumulative_Degradation']
train_data['Lap_to_Progress'] = train_data['LapNumber'] / train_data['RaceProgress']
test_data['Lap_to_Progress'] = test_data['LapNumber'] / test_data['RaceProgress']

train_data.replace([np.inf, -np.inf], 0, inplace=True)
test_data.replace([np.inf, -np.inf], 0, inplace=True)
train_data.fillna(0, inplace=True)
test_data.fillna(0, inplace=True)

train_data = train_data.drop(['Year','Race','Driver','Position'],axis=1)
test_data = test_data.drop(['Year','Race','Driver','Position'],axis=1)
train_data['Cumulative_Degradation'] = np.clip(train_data['Cumulative_Degradation'], -200, 200)
test_data['Cumulative_Degradation'] = np.clip(test_data['Cumulative_Degradation'], -200, 200)
train_data['LapTime_Delta'] = np.clip(train_data['LapTime_Delta'], -200, 200)
test_data['LapTime_Delta'] = np.clip(test_data['LapTime_Delta'], -200, 200)
train_data['LapTime (s)'] = np.clip(train_data['LapTime (s)'], 0, 300)
test_data['LapTime (s)'] = np.clip(test_data['LapTime (s)'], 0, 300)



# ==================== ADD K-MEANS FEATURES ====================
cluster_features = ['RaceProgress', 'Stint', 'TyreLife', 'LapNumber']

# 1. Scaling for K-Means
kmeans_scaler = StandardScaler()
train_scaled_km = kmeans_scaler.fit_transform(train_data[cluster_features])
test_scaled_km = kmeans_scaler.transform(test_data[cluster_features]) 

# 2. Training K-Means in Train
kmeans = KMeans(n_clusters=5, random_state=42)

train_distances = kmeans.fit_transform(train_scaled_km) 
test_distances = kmeans.transform(test_scaled_km)

# 3. Add new columns to DataFrames
centroid_cols = ['Centroid_0', 'Centroid_1', 'Centroid_2', 'Centroid_3', 'Centroid_4']

for i, col_name in enumerate(centroid_cols):
    train_data[col_name] = train_distances[:, i]
    test_data[col_name] = test_distances[:, i]
# ===================================================================



# Create y and X
# Remove rows with missing target, separate target from predictors
train_data.dropna(axis=0, subset=['PitNextLap'], inplace=True)
y = train_data.PitNextLap
X = train_data.drop(['PitNextLap'], axis=1)

# Split into validation and training data
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=1)


# =================== Preproccessing =========================

# --- Data exploration for number of unique entries ---

# Select categorical columns 
categorical_cols = [cname for cname in X.columns if X[cname].dtype == "object"]

# Select numerical columns
numerical_cols = [cname for cname in X.columns if
                X[cname].dtype in ['int64', 'float64']]


# Get number of unique entries in each column with categorical data
categorical_nunique = list(map(lambda col: X[col].nunique(), categorical_cols))
d = dict(zip(categorical_cols, categorical_nunique))
# Print number of unique entries by column, in ascending order
print(f'Num of unique entries by column\n{sorted(d.items(), key=lambda x: x[1])}')


# Select low cardinality columns (< 10 unique values)
low_cardinality_cols = [cname for cname in categorical_cols if 
                        X[cname].nunique() < 10]

# Select high cardinality columns (>= 10 unique values)
high_cardinality_cols = [cname for cname in categorical_cols if 
                         X[cname].nunique() >= 10]


# Keep selected columns only
my_cols = low_cardinality_cols + high_cardinality_cols + numerical_cols
X_final = X[my_cols].copy()
X_test = test_data[my_cols].copy()



# ----------- Transforming --------------

# Transforming for OneHot
oh_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Transforming for Ordinal (High Cardinality)
ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('oh', oh_transformer, low_cardinality_cols),
        ('ord', ordinal_transformer, high_cardinality_cols)
    ],
    remainder='passthrough' # This adds numerical cols to preproccessor
)



# ========== Define XGBoost model==============
# Preproccessing
#X_train_prepped = preprocessor.fit_transform(X_train)
#Χ_val_prepped = preprocessor.transform(X_val)

# Define model and fit
#test_xgb = XGBClassifier(n_estimators=1500, learning_rate=0.02, eval_metric='auc', scale_pos_weight = 10, early_stopping_rounds=50, random_state=0)
#test_xgb.fit(X_train_prepped, y_train, 
#             eval_set=[(X_val_prepped, y_val)], 
#             verbose=True)


# ----------- Run final model ---------------
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(n_estimators=932, learning_rate=0.02, eval_metric='auc', scale_pos_weight = 10, random_state=0))
])

# Cross validation score
scores = cross_val_score(final_pipeline, X_final, y, cv=5, scoring='roc_auc')
print(f"ROC-AUC score: {scores.mean():,.2f}")


# Model training
final_pipeline.fit(X_final, y)



# ======== Define the model for test data ========

# Preprocessing of test data, fit model
test_preds_probs = final_pipeline.predict_proba(X_test)[:, 1] # Possibilities for class 1

# Save test predictions to file
output = pd.DataFrame({
    'id': test_data['id'].values.flatten(),  # Conversion to a simple 1-dimensional array
    'PitNextLap': test_preds_probs.flatten() # Ensuring it is 1 dimension
})
output.to_csv('submission.csv', index=False)


/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv
Num of unique entries by column
[('Compound', 5)]
ROC-AUC score: 0.94
